# 25장 공개 언어모델 다루기 — 불러오고, 얼리고, 조금만 학습하기

**2단계 · 주석 달기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_2_comment.ipynb)

이 노트북에는 **주석을 지운 코드**가 들어 있습니다. 아래 표의 「주석 달기」 과제대로 내 말로 주석을 달아 보세요. 주석을 다는 동안 코드를 한 줄씩 다시 읽게 됩니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 셀을 실행하면 내가 고친 내용이 파일에 저장됩니다.
- 그 아래 확인 셀을 실행해, 주석을 단 뒤에도 코드가 전과 똑같이 도는지 봅니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01`, `p02` 를 치고 실행(`p02` 는 10분 이상) | [ch25_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_1_follow.ipynb) |
| 2. 주석 달기 | `p02` 의 A·B에 "얼린 것 / 학습하는 것 / 그 개수"를 주석으로 | [ch25_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_2_comment.ipynb) **(지금 이 노트북)** |
| 3. 다시 만들기 | 18장 `p02_finetune.py` 와 이 장 `p02` 를 나란히 놓고 같은 구조로 정리한 뒤, 14장 패키지에 `--model koelectra --lora` 로 등록 | [ch25_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch25_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "25"))                      # 이 장의 코드 폴더로 이동
if "google.colab" in sys.modules:                    # Colab: peft 설치, 충돌하는 옛 torchao 제거 (25장 「자주 막히는 곳」)
    !pip install -q peft
    !pip uninstall -y -q torchao
os.environ["EPOCHS"] = "1"                           # 확인은 1 에포크만
print("준비 끝. 작업 폴더: code/25")

A·B에 「얼린 것 / 학습하는 것 / 그 개수」를 주석으로 다세요.

## `p02_finetune_lora.py` — 대상: 파일 전체

In [ ]:
%%writefile p02_finetune_lora.py
import os
import sys
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "19"))
from statutes_dataset import StatuteDataset

NAME = "monologg/koelectra-small-v3-discriminator"
LAWS = ["민법", "형법", "민사소송법", "형사소송법"]
tok = AutoTokenizer.from_pretrained(NAME)
torch.manual_seed(0)
raw_train = StatuteDataset(LAWS, "train")
raw_test = StatuteDataset(LAWS, "test", vocab=raw_train.vocab)


class HFDataset(Dataset):
    """조문 원문을 HF 토크나이저로 인코딩. 15·19장의 Dataset 뼈대 그대로."""

    def __init__(self, items, max_len=64):
        self.items, self.max_len = items, max_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        text, label = self.items[i]
        enc = tok(text, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        return enc["input_ids"][0], enc["attention_mask"][0], label


train_loader = DataLoader(HFDataset(raw_train.items), batch_size=32, shuffle=True)
test_loader = DataLoader(HFDataset(raw_test.items), batch_size=128)


def accuracy(model):
    model.eval()
    c = 0
    with torch.no_grad():
        for ids, mask, y in test_loader:
            c += (model(input_ids=ids, attention_mask=mask).logits.argmax(1) == y).sum().item()
    model.train()
    return c / len(raw_test)


def run(name, model, params, lr, epochs=int(os.environ.get("EPOCHS", 2))):
    opt = torch.optim.AdamW(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n_train = sum(p.numel() for p in params)
    n_all = sum(p.numel() for p in model.parameters())
    t0 = time.time()
    for ep in range(1, epochs + 1):
        for ids, mask, y in train_loader:
            opt.zero_grad()
            loss_fn(model(input_ids=ids, attention_mask=mask).logits, y).backward()
            opt.step()
        print(f"  {name} 에포크 {ep}: 시험 정확도 {accuracy(model):.3f}  ({time.time() - t0:.0f}초)")
    print(f"{name:<18} 학습 파라미터 {n_train:>10,} / 전체 {n_all:,} ({100 * n_train / n_all:.2f}%)")


if __name__ == "__main__":
    torch.manual_seed(0)
    m = AutoModelForSequenceClassification.from_pretrained(NAME, num_labels=len(LAWS))
    for p in m.parameters():
        p.requires_grad = False
    for p in m.classifier.parameters():
        p.requires_grad = True
    run("A. 분류층만", m, list(m.classifier.parameters()), lr=1e-3)

    torch.manual_seed(0)
    m = AutoModelForSequenceClassification.from_pretrained(NAME, num_labels=len(LAWS))
    cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["query", "value"], lora_dropout=0.1,
                     task_type="SEQ_CLS")
    m = get_peft_model(m, cfg)
    trainable = [p for p in m.parameters() if p.requires_grad]
    run("B. LoRA(r=8)", m, trainable, lr=5e-4)
    m.print_trainable_parameters()

### 확인

In [ ]:
!python p02_finetune_lora.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- p02_finetune_lora.py